In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']

        
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    
    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-04 19:40:47,998 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp5ypdcr3_
2023-08-04 19:40:47,999 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp5ypdcr3_/_remote_module_non_scriptable.py


In [2]:
# THE other FASST extrinisc15 dataset had random state = 4

countlen = len(df)*0.7*0.925

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 100 # Number of bootstrapping iterations

# Placeholder for the results
results = []
syn_model = Plugins().get('ctgan')
# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset
    start_time = time.time()    
    
    # Set the size of your bootstrap sample. This could be the size of your original dataset
    bootstrap_size = int(0.7 * df.shape[0])

    # Perform bootstrapping
    df_train_main = resample(df, replace=True, n_samples=bootstrap_size, random_state=i*124)

    # Find the Out-of-Bag samples
    oob_index = df.index.difference(df_train_main.index)
    df_test = df.loc[oob_index]

    # Now split the training set into two subsets
    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.075, random_state=i*1244)

    # Resample
    #df_test_resample = resample(df_test,replace=True)
    #df_train_1_resample = resample(df_train_1,replace=True)
    #df_train_2_resample = resample(df_train_2,replace=True)
    
    loader = GenericDataLoader(df_train_1, target_column='Response')
   
    syn_model.fit(loader)
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()

        print(syn_set['Response'].mean() * 100)

        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 30%|███████████▉                            | 599/2000 [06:24<15:00,  1.56it/s]


0.0
10.238429172510518
10.441485634197617
9.15246322671025
9.58251611095545
9.867787409158568
9.601914545879056
9.619051324954278
9.60886028848469
9.60003502064759
9.744459644322845
Time: 716.2824289798737 seconds
Iteration: 0 


 50%|███████████████████▉                    | 999/2000 [10:08<10:09,  1.64it/s]


0.0
10.799438990182328
10.2312543798178
9.666121877188887
10.703278229195853
10.253042640749497
10.588372636002802
10.599634227012725
10.59163438177162
10.3850812041266
10.37920656634747
Time: 955.7399091720581 seconds
Iteration: 1 


 27%|██████████▉                             | 549/2000 [05:44<15:11,  1.59it/s]


...

0.0
18.092566619915846
17.449194113524875
17.76791968246556
16.797422247128047
16.776114175641364
16.822320803175344
17.020117514300168
16.672138682776282
16.59832776407757
16.716826265389876
Time: 657.2582359313965 seconds
Iteration: 97 


 40%|███████████████▉                        | 799/2000 [05:28<08:13,  2.43it/s]


0.0
14.305750350631136
13.17449194113525
14.475834695307027
14.555898010647239
13.895455739427371
13.798739201494278
13.973306354332854
14.080591853262417
14.00242226145832
13.929411764705883
Time: 622.2077450752258 seconds
Iteration: 98 


 35%|█████████████▉                          | 699/2000 [04:53<09:05,  2.38it/s]


100.0
15.287517531556801
16.1177295024527
14.592575297688537
14.233678901653125
14.806059014096839
14.726826990427272
14.876065216545392
14.612471819117035
14.749529410048007
14.70205198358413
Time: 553.3170297145844 seconds
Iteration: 99 


In [3]:
results_exc_df_1.to_clipboard()